In [12]:
!pip install rdflib

In [13]:
import os
import sys
import pandas as pd
from rdflib import Graph, Literal, Namespace, RDF
from rdflib.namespace import XSD
from pathlib import Path

root = Path.cwd().parent
sys.path.insert(0, str(root))
from config import DATA_DIR, DATA_DIR_RAW

In [14]:
# Read the synthetic data
suppliers_df = pd.read_csv(DATA_DIR_RAW / 'suppliers.csv')
products_df = pd.read_csv(DATA_DIR_RAW / 'products.csv')
warehouses_df = pd.read_csv(DATA_DIR_RAW / 'warehouses.csv')
customers_df = pd.read_csv(DATA_DIR_RAW / 'customers.csv')
routes_df = pd.read_csv(DATA_DIR_RAW / 'routes.csv')
orders_df = pd.read_csv(DATA_DIR_RAW / 'orders.csv')
inventory_df = pd.read_csv(DATA_DIR_RAW / 'inventory.csv')
shipments_df = pd.read_csv(DATA_DIR_RAW / 'shipments.csv')
logistics_events_df = pd.read_csv(DATA_DIR_RAW / 'logistics_events.csv')

In [15]:
# ---------- Output folder ----------
os.makedirs(f"{DATA_DIR}/processed", exist_ok=True)

# ---------- Graph ----------
g = Graph()

# ---------- Namespaces ----------
ex = Namespace("http://example.org/supplychain/")
g.bind("ex", ex)
g.bind("rdf", RDF)
g.bind("xsd", XSD)

In [16]:
# ---------- Suppliers ----------
for _, row in suppliers_df.iterrows():
    supplier_node = ex[f"supplier/{int(row['SupplierID'])}"]
    g.add((supplier_node, RDF.type, ex.Supplier))
    g.add((supplier_node, ex.name, Literal(row["Name"])))
    g.add((supplier_node, ex.country, Literal(row["Country"])))
    g.add((supplier_node, ex.reliabilityScore, Literal(float(row["ReliabilityScore"]), datatype=XSD.float)))
    g.add((supplier_node, ex.leadTime, Literal(int(row["LeadTimeDays"]), datatype=XSD.integer)))

# ---------- Products ----------
for _, row in products_df.iterrows():
    product_node = ex[f"product/{int(row['ProductID'])}"]
    supplier_node = ex[f"supplier/{int(row['SupplierID'])}"]

    g.add((product_node, RDF.type, ex.Product))
    g.add((product_node, ex.sku, Literal(row["SKU"])))
    g.add((product_node, ex.category, Literal(row["Category"])))
    g.add((product_node, ex.demandLevel, Literal(float(row["DemandLevel"]), datatype=XSD.float)))
    g.add((product_node, ex.stockQty, Literal(int(row["StockQty"]), datatype=XSD.integer)))
    g.add((product_node, ex.supplier, supplier_node))

# ---------- Warehouses ----------
for _, row in warehouses_df.iterrows():
    warehouse_node = ex[f"warehouse/{int(row['WarehouseID'])}"]
    g.add((warehouse_node, RDF.type, ex.Warehouse))
    g.add((warehouse_node, ex.location, Literal(row["Location"])))
    g.add((warehouse_node, ex.capacity, Literal(int(row["Capacity"]), datatype=XSD.integer)))
    g.add((warehouse_node, ex.currentLoad, Literal(int(row["CurrentLoad"]), datatype=XSD.integer)))

# ---------- Customers ----------
for _, row in customers_df.iterrows():
    customer_node = ex[f"customer/{int(row['CustomerID'])}"]
    g.add((customer_node, RDF.type, ex.Customer))
    g.add((customer_node, ex.region, Literal(row["Region"])))
    g.add((customer_node, ex.tier, Literal(int(row["Tier"]), datatype=XSD.integer)))

# ---------- Routes ----------
for _, row in routes_df.iterrows():
    route_node = ex[f"route/{int(row['RouteID'])}"]
    origin_node = ex[f"warehouse/{int(row['OriginWarehouseID'])}"]
    destination_node = ex[f"warehouse/{int(row['DestinationWarehouseID'])}"]

    g.add((route_node, RDF.type, ex.Route))
    g.add((route_node, ex.originWarehouse, origin_node))
    g.add((route_node, ex.destinationWarehouse, destination_node))
    g.add((route_node, ex.avgTransitDays, Literal(int(row["AvgTransitDays"]), datatype=XSD.integer)))
    g.add((route_node, ex.cost, Literal(float(row["Cost"]), datatype=XSD.float)))

# ---------- Orders ----------
for _, row in orders_df.iterrows():
    order_node = ex[f"order/{int(row['OrderID'])}"]
    customer_node = ex[f"customer/{int(row['CustomerID'])}"]
    product_node = ex[f"product/{int(row['ProductID'])}"]
    warehouse_node = ex[f"warehouse/{int(row['WarehouseID'])}"]

    g.add((order_node, RDF.type, ex.Order))
    g.add((order_node, ex.status, Literal(row["Status"])))
    g.add((order_node, ex.priority, Literal(int(row["Priority"]), datatype=XSD.integer)))
    g.add((order_node, ex.date, Literal(pd.Timestamp(row["Date"]).date().isoformat(), datatype=XSD.date)))
    g.add((order_node, ex.customer, customer_node))
    g.add((order_node, ex.product, product_node))
    g.add((order_node, ex.warehouse, warehouse_node))

# ---------- Inventory ----------
for _, row in inventory_df.iterrows():
    inventory_node = ex[f"inventory/{int(row['InventoryID'])}"]
    warehouse_node = ex[f"warehouse/{int(row['WarehouseID'])}"]
    product_node = ex[f"product/{int(row['ProductID'])}"]

    g.add((inventory_node, RDF.type, ex.Inventory))
    g.add((inventory_node, ex.warehouse, warehouse_node))
    g.add((inventory_node, ex.product, product_node))
    g.add((inventory_node, ex.stockOnHand, Literal(int(row["StockOnHand"]), datatype=XSD.integer)))
    g.add((inventory_node, ex.reservedQty, Literal(int(row["ReservedQty"]), datatype=XSD.integer)))
    g.add((inventory_node, ex.reorderPoint, Literal(int(row["ReorderPoint"]), datatype=XSD.integer)))

# ---------- Shipments ----------
for _, row in shipments_df.iterrows():
    shipment_node = ex[f"shipment/{int(row['ShipmentID'])}"]
    order_node = ex[f"order/{int(row['OrderID'])}"]
    route_node = ex[f"route/{int(row['RouteID'])}"]
    from_warehouse_node = ex[f"warehouse/{int(row['FromWarehouseID'])}"]
    to_warehouse_node = ex[f"warehouse/{int(row['ToWarehouseID'])}"]

    g.add((shipment_node, RDF.type, ex.Shipment))
    g.add((shipment_node, ex.shippedDate, Literal(str(row["ShippedDate"]), datatype=XSD.date)))
    g.add((shipment_node, ex.expectedDeliveryDate, Literal(str(row["ExpectedDeliveryDate"]), datatype=XSD.date)))
    g.add((shipment_node, ex.actualDeliveryDate, Literal(str(row["ActualDeliveryDate"]), datatype=XSD.date)))
    g.add((shipment_node, ex.shipmentStatus, Literal(row["ShipmentStatus"])))
    g.add((shipment_node, ex.order, order_node))
    g.add((shipment_node, ex.route, route_node))
    g.add((shipment_node, ex.fromWarehouse, from_warehouse_node))
    g.add((shipment_node, ex.toWarehouse, to_warehouse_node))

# ---------- Logistics Events ----------
for _, row in logistics_events_df.iterrows():
    event_node = ex[f"event/{int(row['EventID'])}"]
    g.add((event_node, RDF.type, ex.LogisticsEvent))
    g.add((event_node, ex.eventType, Literal(row["EventType"])))
    g.add((event_node, ex.cause, Literal(row["Cause"])))

    if pd.notna(row["DelayDays"]) and int(row["DelayDays"]) > 0:
        g.add((event_node, ex.delayDays, Literal(int(row["DelayDays"]), datatype=XSD.integer)))

    if pd.notna(row["SupplierID"]):
        g.add((event_node, ex.supplier, ex[f"supplier/{int(row['SupplierID'])}"]))

    if pd.notna(row["RouteID"]):
        g.add((event_node, ex.route, ex[f"route/{int(row['RouteID'])}"]))

    if pd.notna(row["WarehouseID"]):
        g.add((event_node, ex.warehouse, ex[f"warehouse/{int(row['WarehouseID'])}"]))

    if pd.notna(row["OrderID"]):
        g.add((event_node, ex.order, ex[f"order/{int(row['OrderID'])}"]))

    if pd.notna(row["ShipmentID"]):
        g.add((event_node, ex.shipment, ex[f"shipment/{int(row['ShipmentID'])}"]))

In [17]:
# ---------- Serialize ----------
g.serialize(destination=f"{DATA_DIR}/processed/supplychain_kg.ttl", format="turtle")

print("Knowledge graph generated and saved to data/processed/supplychain_kg.ttl")

Knowledge graph generated and saved to data/processed/supplychain_kg.ttl
